# Module 3.7 — SOLID Principles & Architecture

### Python Mastery · Track 3: Object-Oriented Programming & Design Patterns

SOLID is a set of five principles for designing object-oriented code that is easy to understand, extend, and maintain. This module explains each principle with a contrasting before-and-after example, then makes the practical case for favouring composition over inheritance.

**How to use this notebook**

- Read each explanation, then run the code cell beneath it.
- For each principle, study the weaker design first, then the improved one.
- Attempt the exercises before consulting the solutions.

### Learning objectives

After completing this module you will be able to:

1. Apply the Single Responsibility Principle to keep classes focused.
2. Apply the Open-Closed Principle to extend behaviour without editing existing code.
3. Apply the Liskov Substitution Principle so subclasses are safe replacements.
4. Apply Interface Segregation and Dependency Inversion.
5. Choose composition over inheritance where it gives more flexibility.

**Prerequisites:** Tracks 1 and 2, and Modules 3.1 to 3.6.

---

## Part 1 · Single Responsibility Principle (SRP)

A class should have one reason to change, meaning one clear responsibility. When a class does several unrelated things, a change to one concern risks breaking the others and the class becomes hard to test. The fix is to split responsibilities into focused classes.

In [ ]:
# Weaker design: one class mixes data, formatting, and persistence.
class ReportBad:
    def __init__(self, data):
        self.data = data
    def format_html(self):
        return f"<p>{self.data}</p>"
    def save_to_file(self, path):
        # Mixing presentation with file handling: two reasons to change.
        return f"would write {self.format_html()} to {path}"

# Improved: each class has a single responsibility.
class Report:
    def __init__(self, data):
        self.data = data

class HtmlFormatter:
    def format(self, report):
        return f"<p>{report.data}</p>"

class FileSaver:
    def save(self, content, path):
        return f"would write {content!r} to {path}"

report = Report("Quarterly results")
html = HtmlFormatter().format(report)
print(FileSaver().save(html, "report.html"))
# Now formatting and saving can change independently and be tested in isolation.

## Part 2 · Open-Closed Principle (OCP)

Software should be open for extension but closed for modification: you should be able to add new behaviour without editing tested, working code. A growing chain of `if`/`elif` on a type is the classic warning sign; replacing it with polymorphism lets new cases be added as new classes.

In [ ]:
from abc import ABC, abstractmethod

# Weaker design: adding a shape means editing this function every time.
def area_bad(shape):
    if shape["type"] == "circle":
        return 3.14159 * shape["r"] ** 2
    elif shape["type"] == "square":
        return shape["side"] ** 2
    # each new shape forces a change here

# Improved: a common interface; new shapes are new classes, no edits needed.
class Shape(ABC):
    @abstractmethod
    def area(self):
        ...

class Circle(Shape):
    def __init__(self, r): self.r = r
    def area(self): return 3.14159 * self.r ** 2

class Square(Shape):
    def __init__(self, side): self.side = side
    def area(self): return self.side ** 2

class Triangle(Shape):                     # added later WITHOUT touching existing code
    def __init__(self, base, height): self.base, self.height = base, height
    def area(self): return 0.5 * self.base * self.height

shapes = [Circle(2), Square(3), Triangle(4, 5)]
print("areas:", [round(s.area(), 2) for s in shapes])

## Part 3 · Liskov Substitution Principle (LSP)

Objects of a subclass should be usable anywhere the base class is expected, without surprising the caller. A subclass that weakens guarantees or changes expected behaviour violates this and causes subtle bugs. The classic example is forcing an inheritance that does not truly hold.

In [ ]:
# Weaker design: a Penguin "is a" Bird that can fly, but penguins cannot.
class BirdBad:
    def fly(self):
        return "flying"

class PenguinBad(BirdBad):
    def fly(self):
        raise NotImplementedError("penguins cannot fly")   # breaks the contract

# Improved: model the real capability, not a forced hierarchy.
class Bird:
    def move(self):
        return "moving"

class FlyingBird(Bird):
    def move(self):
        return "flying"

class Penguin(Bird):
    def move(self):
        return "swimming"

def travel(bird: Bird):
    return bird.move()                     # works for ANY Bird, no exceptions

for b in [FlyingBird(), Penguin()]:
    print(type(b).__name__, "->", travel(b))

## Part 4 · Interface Segregation and Dependency Inversion

**Interface Segregation (ISP):** clients should not be forced to depend on methods they do not use. Prefer several small, focused interfaces over one large one, so a class implements only what is relevant to it.

**Dependency Inversion (DIP):** high-level code should depend on abstractions, not on concrete details. By depending on an interface and receiving the concrete implementation from outside (injection), you can swap implementations freely, which also makes testing easy.

In [ ]:
from abc import ABC, abstractmethod

# Interface Segregation: small, focused protocols rather than one fat interface.
class Printer(ABC):
    @abstractmethod
    def print_doc(self, doc): ...

class Scanner(ABC):
    @abstractmethod
    def scan(self): ...

# A simple printer implements only what it can do, not scanning it lacks.
class BasicPrinter(Printer):
    def print_doc(self, doc):
        return f"printing {doc}"

print(BasicPrinter().print_doc("letter"))

In [ ]:
from abc import ABC, abstractmethod

# Dependency Inversion: the high-level class depends on an abstraction.
class MessageSender(ABC):
    @abstractmethod
    def send(self, message): ...

class EmailSender(MessageSender):
    def send(self, message): return f"email: {message}"

class SmsSender(MessageSender):
    def send(self, message): return f"sms: {message}"

class Notifier:
    def __init__(self, sender: MessageSender):
        self.sender = sender               # the concrete sender is injected in
    def alert(self, text):
        return self.sender.send(text)

# We can swap implementations without changing Notifier at all.
print(Notifier(EmailSender()).alert("server down"))
print(Notifier(SmsSender()).alert("server down"))

## Part 5 · Composition Over Inheritance

Inheritance expresses an "is a" relationship and is powerful, but deep or wide hierarchies become rigid: a change high up ripples everywhere, and a class is locked into its parents. **Composition** builds behaviour by holding other objects ("has a"), which is more flexible because parts can be combined and swapped at runtime. The guidance is to prefer composition unless a genuine "is a" relationship and shared interface justify inheritance.

In [ ]:
# Inheritance approach: behaviour is fixed by the class hierarchy.
class LoggingMixin:
    def log(self, msg): return f"[log] {msg}"

# Composition approach: behaviour is assembled from independent parts.
class Logger:
    def log(self, msg): return f"[log] {msg}"

class Encryptor:
    def encrypt(self, data): return data[::-1]   # stand-in transformation

class Service:
    """Composes a logger and an encryptor rather than inheriting them."""
    def __init__(self, logger, encryptor):
        self.logger = logger
        self.encryptor = encryptor
    def process(self, data):
        secured = self.encryptor.encrypt(data)
        return self.logger.log(f"processed -> {secured}")

# The parts can be swapped independently, which inheritance would not allow as easily.
service = Service(Logger(), Encryptor())
print(service.process("hello"))

---

## Worked Examples

| Examples | Level |
|---|---|
| 1 and 2 | Easy |
| 3 and 4 | Medium |
| 5 and 6 | Difficult |

### Example 1 — Single Responsibility in practice (Easy)

In [ ]:
# Separate calculating from displaying.
class Calculator:
    def add(self, a, b):
        return a + b

class Display:
    def show(self, value):
        return f"Result: {value}"

total = Calculator().add(2, 3)
print(Display().show(total))
# Each class does one thing; either can change without affecting the other.

### Example 2 — Open-Closed with a common interface (Easy)

In [ ]:
from abc import ABC, abstractmethod

class Greeting(ABC):
    @abstractmethod
    def text(self): ...

class English(Greeting):
    def text(self): return "Hello"

class French(Greeting):
    def text(self): return "Bonjour"

# A new language is a new class; this loop never changes.
for g in [English(), French()]:
    print(g.text())

### Example 3 — Dependency injection for testability (Medium)

In [ ]:
class Database:
    def get_user(self, uid):
        return f"real user {uid}"

class FakeDatabase:
    def get_user(self, uid):
        return f"test user {uid}"          # a stand-in used during testing

class UserService:
    def __init__(self, db):
        self.db = db                        # depend on whatever is injected
    def greet(self, uid):
        return f"Welcome, {self.db.get_user(uid)}"

print(UserService(Database()).greet(1))
print(UserService(FakeDatabase()).greet(1))   # easy to test without a real database

### Example 4 — Liskov-safe hierarchy (Medium)

In [ ]:
class Account:
    def __init__(self, balance):
        self.balance = balance
    def withdraw(self, amount):
        if amount > self.balance:
            return "declined"
        self.balance -= amount
        return "ok"

class SavingsAccount(Account):
    # Does not break the parent contract: still returns "ok" or "declined".
    def withdraw(self, amount):
        result = super().withdraw(amount)
        return result

def drain(account, amount):
    return account.withdraw(amount)         # behaves correctly for either type

print(drain(Account(100), 50))
print(drain(SavingsAccount(100), 150))

### Example 5 — Composition to combine behaviours (Difficult)

In [ ]:
# Build a document processor by composing independent strategies.
class PlainText:
    def render(self, text): return text

class Bold:
    def render(self, text): return f"**{text}**"

class Uppercase:
    def render(self, text): return text.upper()

class Document:
    def __init__(self, renderers):
        self.renderers = renderers          # a list of composed behaviours
    def render(self, text):
        for renderer in self.renderers:
            text = renderer.render(text)
        return text

# Combine and reorder behaviours freely, without any inheritance.
doc1 = Document([Uppercase(), Bold()])
doc2 = Document([Bold(), PlainText()])
print(doc1.render("hello"))
print(doc2.render("hello"))

### Example 6 — A small layered design (Difficult)

In [ ]:
from abc import ABC, abstractmethod

# Abstraction the high-level code depends on (Dependency Inversion).
class Repository(ABC):
    @abstractmethod
    def all(self): ...

# A concrete detail, easily swapped.
class InMemoryRepository(Repository):
    def __init__(self):
        self._items = ["alpha", "beta", "gamma"]
    def all(self):
        return list(self._items)

# A focused service (Single Responsibility) that receives its dependency (injection).
class SearchService:
    def __init__(self, repository: Repository):
        self.repository = repository
    def starting_with(self, prefix):
        return [item for item in self.repository.all() if item.startswith(prefix)]

service = SearchService(InMemoryRepository())
print("items with 'a' or 'b':", service.starting_with("a") + service.starting_with("b"))

---

## Exercises

**Exercise 1 (Easy).** A class currently both computes a total and prints it. Split it into a `Cart` class (with a `total()` method over a list of prices) and a `Receipt` class (with a `show(total)` method). Demonstrate them working together.

In [ ]:
# Your solution here


**Exercise 2 (Medium).** Using an abstract base class, design a `Discount` interface with a method `apply(price)`, and two implementations (`NoDiscount`, `PercentageDiscount`). Write a `checkout(price, discount)` function that works with any discount, so new discounts need no change to checkout.

In [ ]:
# Your solution here


**Exercise 3 (Medium).** Apply Dependency Inversion: write a `Report` class that depends on an injected `formatter` object with a `format(text)` method. Provide two formatters and show the same report rendered both ways.

In [ ]:
# Your solution here


**Exercise 4 (Medium).** Demonstrate Liskov substitution: define a `Shape` base with an `area()` method and two subclasses, then write a function `print_areas(shapes)` that works for a mixed list without any type checks.

In [ ]:
# Your solution here


**Exercise 5 (Difficult).** Use composition to build a `Notifier` that holds a list of channel objects (each with a `send(message)` method) and sends a message through all of them. Provide two channels and demonstrate a broadcast.

In [ ]:
# Your solution here


---

## Solutions

**Solution 1**

In [ ]:
class Cart:
    def __init__(self, prices):
        self.prices = prices
    def total(self):
        return sum(self.prices)

class Receipt:
    def show(self, total):
        return f"Total due: {total}"

cart = Cart([100, 250, 75])
print(Receipt().show(cart.total()))

**Solution 2**

In [ ]:
from abc import ABC, abstractmethod

class Discount(ABC):
    @abstractmethod
    def apply(self, price): ...

class NoDiscount(Discount):
    def apply(self, price): return price

class PercentageDiscount(Discount):
    def __init__(self, percent): self.percent = percent
    def apply(self, price): return price * (1 - self.percent / 100)

def checkout(price, discount):
    return discount.apply(price)

print(checkout(200, NoDiscount()))
print(checkout(200, PercentageDiscount(25)))

**Solution 3**

In [ ]:
class Report:
    def __init__(self, text, formatter):
        self.text = text
        self.formatter = formatter
    def render(self):
        return self.formatter.format(self.text)

class PlainFormatter:
    def format(self, text): return text

class HtmlFormatter:
    def format(self, text): return f"<p>{text}</p>"

print(Report("hello", PlainFormatter()).render())
print(Report("hello", HtmlFormatter()).render())

**Solution 4**

In [ ]:
from abc import ABC, abstractmethod

class Shape(ABC):
    @abstractmethod
    def area(self): ...

class Square(Shape):
    def __init__(self, s): self.s = s
    def area(self): return self.s ** 2

class Circle(Shape):
    def __init__(self, r): self.r = r
    def area(self): return 3.14159 * self.r ** 2

def print_areas(shapes):
    for shape in shapes:
        print(type(shape).__name__, round(shape.area(), 2))

print_areas([Square(3), Circle(2)])

**Solution 5**

In [ ]:
class EmailChannel:
    def send(self, message): return f"email: {message}"

class SmsChannel:
    def send(self, message): return f"sms: {message}"

class Notifier:
    def __init__(self, channels):
        self.channels = channels
    def broadcast(self, message):
        return [channel.send(message) for channel in self.channels]

notifier = Notifier([EmailChannel(), SmsChannel()])
print(notifier.broadcast("deployment complete"))

---

## Summary and Key Points

- Single Responsibility: each class should have one reason to change; split mixed concerns into focused classes that can be tested and changed independently.
- Open-Closed: extend behaviour by adding new classes against a common interface rather than editing existing, tested code; replace type-based conditionals with polymorphism.
- Liskov Substitution: a subclass must be a safe replacement for its base, preserving the expected behaviour and contract; do not force inheritance that does not truly hold.
- Interface Segregation favours small, focused interfaces; Dependency Inversion has high-level code depend on abstractions and receive concrete implementations by injection, which also eases testing.
- Prefer composition (has a) over inheritance (is a) for flexibility, combining and swapping parts at runtime; use inheritance only for genuine "is a" relationships with a shared interface.

### Track 3 complete

This concludes Track 3, Object-Oriented Programming & Design Patterns. You can now design classes with appropriate data and methods, build and reason about hierarchies, integrate objects with Python's syntax through magic methods, control attributes with properties and descriptors, model fixed choices with enums and records, apply common design patterns, and structure code with SOLID principles and composition. Track 4 builds on this with functional programming and metaprogramming.